# Driscoll-Kraay Pipeline - Conservative SE Workflow

This notebook mirrors `02_modern_pipeline.ipynb` but refits every regression
with **Driscoll-Kraay (1998) kernel standard errors** instead of the
two-way-clustered default.

## Why

The Pesaran (2004) CD test on our baseline residuals rejects the null of
cross-sectional independence for every globalisation index (see
`outputs/tables/residual_cd_test.tex`). Under strong CSD:

- **One-way entity clustering** (literature convention) is anti-conservative
  because it assumes errors are independent across countries.
- **Two-way clustering** (entity + time) handles *weak* CSD but becomes
  inconsistent with only T ~ 40 time periods.
- **Driscoll-Kraay** kernel SEs are nonparametric and robust to arbitrary
  cross-sectional dependence plus serial correlation. They are the
  econometrically defensible default when a CD test rejects independence in
  a macro panel.

## Scope

Every regression in this notebook uses `run_panel_ols(..., cov_type="kernel")`.
Point estimates are identical to the two-way-clustered pipeline — only the
standard errors, t-statistics, p-values, and significance stars change.

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from linearmodels.panel import compare

from analysis.regression_utils import (
    LATEX_LABEL_MAP,
    generate_marginal_effects,
    prepare_regression_data,
    run_panel_ols,
    significance_stars,
)
from clean.panel_utils import add_welfare_regimes, create_lags
from clean.utils import load_config

config = load_config()
master = add_welfare_regimes(
    pd.read_parquet(REPO_ROOT / "data" / "final" / "master_dataset.parquet")
)

indices = config.get("indices", ["KOFGI", "KOFEcGI", "KOFSoGI", "KOFPoGI"])
ctrls = config.get(
    "controls",
    ["ln_gdppc", "inflation_cpi", "deficit", "debt", "ln_population", "dependency_ratio"],
)
dep_var = config.get("dependent_var", "sstran")

DK_TABLES_DIR = REPO_ROOT / "outputs" / "tables" / "dk"
DK_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Master panel: {master.shape[0]} rows, {master['iso3'].nunique()} countries, "
      f"{master['year'].min()}-{master['year'].max()}")
print(f"Driscoll-Kraay tables will be written to: {DK_TABLES_DIR}")

## Helper: fit PanelOLS with Driscoll-Kraay SEs

Wraps `run_panel_ols(..., cov_type="kernel")` so every call in this notebook
uses DK kernel SEs. The function returns a `PanelResults` object that behaves
like any other `linearmodels` fit — same `.params`, `.std_errors`, `.pvalues`,
`.summary` — so we can feed it straight into `compare(...)` for side-by-side
tables and into `generate_marginal_effects(...)` for regime decomposition.

In [ ]:
def fit_dk(ols_data, dep_var, exog_vars):
    """Fit PanelOLS with entity+time FEs and Driscoll-Kraay SEs."""
    return run_panel_ols(ols_data, dep_var, exog_vars, cov_type="kernel")


def _summarise(res, indep_var):
    return {
        "Coef": round(float(res.params[indep_var]), 4),
        "DK SE": round(float(res.std_errors[indep_var]), 4),
        "t-stat": round(float(res.tstats[indep_var]), 2),
        "p-value": round(float(res.pvalues[indep_var]), 4),
        "Stars": significance_stars(res.pvalues[indep_var]),
        "N": int(res.nobs),
    }

## Part 1: Baseline regressions (no interactions)

The core specification:

$$
\text{sstran}_{it} = \alpha_i + \lambda_t + \beta\,\text{KOF}_{i,t-1} + \gamma' X_{i,t-1} + \varepsilon_{it}
$$

refit per globalisation index with Driscoll-Kraay SEs.

In [ ]:
baseline_models = {}
baseline_summary_rows = []

for idx_name in indices:
    reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, dep_var, indep, lagged_ctrls, interactions=False
    )

    res = fit_dk(ols_data, dep_var, exog)
    baseline_models[idx_name] = res

    row = {"Index": idx_name, **_summarise(res, indep)}
    baseline_summary_rows.append(row)

baseline_summary = pd.DataFrame(baseline_summary_rows)
baseline_summary

In [ ]:
# Full linearmodels side-by-side table (all coefficients, not just the main index)
baseline_comparison = compare(baseline_models, stars=True)
print(baseline_comparison)

In [ ]:
# Persist LaTeX
latex_str = baseline_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "baseline_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

## Part 2: Heterogeneity across welfare regimes

Regime-interaction specification with social-democratic as the reference
category. `prepare_regression_data(interactions=True)` handles the
construction of `int_conservative`, `int_mediterranean`, `int_liberal`, and
`int_post_communist` terms.

In [ ]:
interaction_models = {}

for idx_name in indices:
    reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, dep_var, indep, lagged_ctrls, interactions=True
    )
    interaction_models[idx_name] = fit_dk(ols_data, dep_var, exog)

interaction_comparison = compare(interaction_models, stars=True)
print(interaction_comparison)

In [ ]:
latex_str = interaction_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "interaction_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

### Marginal effects by welfare regime (DK SEs)

`generate_marginal_effects` takes a fitted interaction model and computes
the marginal effect of the lagged globalisation index for each regime, using
linear combinations of the interaction coefficients. Because the fitted
model carries DK standard errors, the derived SEs are DK-consistent too.

In [ ]:
for idx_name, res in interaction_models.items():
    me_df = generate_marginal_effects(res, index_name=idx_name)
    print(f"\n=== Marginal effects ({idx_name}, DK SEs) ===")
    display(me_df)

    out_path = DK_TABLES_DIR / f"marginal_effects_{idx_name}_dk.tex"
    out_path.write_text(me_df.to_latex(index=False, escape=False), encoding="utf-8")

## Part 3: Subperiod splits (China shock & GFC)

Refits the baseline spec within each of four subperiods using DK SEs. This
is where the CSD concern bites hardest — post-break samples are shorter, so
there are fewer time periods over which common shocks can average out.

In [ ]:
SUBPERIODS = {
    "pre_china_shock": (1980, 1999),
    "post_china_shock": (2000, 2023),
    "pre_gfc": (1980, 2007),
    "post_gfc": (2008, 2023),
}

subperiod_rows = []
subperiod_models_by_era = {}

for era, (start, end) in SUBPERIODS.items():
    models_era = {}
    for idx_name in indices:
        reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
        period_data = reg_data[(reg_data["year"] >= start) & (reg_data["year"] <= end)].copy()
        indep = f"{idx_name}_lag1"
        lagged_ctrls = [f"{v}_lag1" for v in ctrls]
        ols_data, exog = prepare_regression_data(
            period_data, dep_var, indep, lagged_ctrls, interactions=False
        )
        if len(ols_data) < len(exog) + 10:
            continue
        res = fit_dk(ols_data, dep_var, exog)
        models_era[idx_name] = res
        subperiod_rows.append({"Period": era, "Index": idx_name, **_summarise(res, indep)})
    if models_era:
        subperiod_models_by_era[era] = models_era

subperiod_summary = pd.DataFrame(subperiod_rows)
subperiod_summary

In [ ]:
for era, models_era in subperiod_models_by_era.items():
    cmp = compare(models_era, stars=True)
    latex_str = cmp.summary.as_latex()
    for old, new in LATEX_LABEL_MAP.items():
        latex_str = latex_str.replace(old, new)
    out_path = DK_TABLES_DIR / f"baseline_regressions_{era}_dk.tex"
    out_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved: {out_path}")

## Part 4: Feedback regressions (reverse causality)

Swaps the dependent and the globalisation variable to test whether welfare
spending predicts future globalisation:

$$
\text{KOF}_{it} = \alpha_i + \lambda_t + \beta\,\text{sstran}_{i,t-1} + \gamma' X_{i,t-1} + \varepsilon_{it}
$$

Same DK SE convention throughout.

In [ ]:
feedback_models = {}
feedback_rows = []

for idx_name in indices:
    feedback_ctrls = [dep_var] + ctrls  # sstran is now a regressor
    reg_data = create_lags(master, [idx_name] + feedback_ctrls, lags=[1])
    indep = f"{dep_var}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, idx_name, indep, lagged_ctrls, interactions=False
    )
    if len(ols_data) < len(exog) + 10:
        continue
    res = fit_dk(ols_data, idx_name, exog)
    feedback_models[idx_name] = res
    feedback_rows.append({"Dependent": idx_name, **_summarise(res, indep)})

feedback_summary = pd.DataFrame(feedback_rows)
feedback_summary

In [ ]:
feedback_comparison = compare(feedback_models, stars=True)
latex_str = feedback_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "feedback_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

## Summary of the DK pipeline

Because switching the covariance estimator does not move the point
estimates, the headline coefficient on the lagged globalisation index is the
same as in `02_modern_pipeline.ipynb`. What changes:

- **Standard errors** — generally larger under DK than under one-way entity
  clustering, because DK allows arbitrary cross-sectional dependence.
- **p-values and significance stars** — correspondingly more conservative.
  Effects that were significant under one-way entity clustering but lose
  their stars here were being propped up by the (violated) independence
  assumption.
- **Economic story** — unchanged in direction, but the range of robust
  conclusions narrows.

## When to prefer which SE

| Context | Recommended SE |
| --- | --- |
| Reviewer familiarity / historical comparability | One-way entity clustering |
| Project-internal default (current) | Two-way clustering |
| Residual CSD present (our case) | **Driscoll-Kraay** |

For the DK-skeptical reader, the side-by-side `se_comparison.tex` and
`se_comparison_{era}.tex` tables (produced by
`export_subperiod_se_comparison_tables`) show all three columns next to each
other — see `03_se_comparison.ipynb`.